# Station Stacking v11 Settlement - KMIA

Experimental notebook for `KMIA`.

This controlled rerun keeps the exact v11 feature/model contract, replaces daily-high labels with settlement-first Wunderground station history when available, and writes separate artifacts to `data/calibration/station_stacking_v11_settlement`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KMIA"
PROVIDERS = ("gfs", "hrrr", "nbm")
TREND_COLUMNS = [
    "observed_temp_change_last_1h_f",
    "observed_temp_change_last_3h_f",
    "observed_morning_warmup_rate_f_per_hour",
    "observed_high_so_far_change_since_9am_f",
]
TIMING_MODE = "same_day_11am_live_safe"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
MODEL_VERSION = "station_high_regressor_v11_wunderground_settlement_stack"
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V11_DROPPED_FEATURE_COLUMNS,
    V11_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V11 Contract

`feature_version="v11"` keeps the v9 feature contract and remaining-warmup target, but trains base learners with Huber-style objectives while retaining the ridge stack selected by validation MAE.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

fold_spec


,fold,train_start_year,train_end_year,validation_year
0,fold_2021_2023_to_2024,2021,2023,2024
1,fold_2021_2024_to_2025,2021,2024,2025


In [4]:
V11_FEATURE_COLUMNS, sorted(V11_DROPPED_FEATURE_COLUMNS)


(['v2_recent_heat_anomaly_f',
  'v2_recent_heat_momentum_f',
  'v2_morning_warmup_to_consensus_f',
  'v2_consensus_minus_7d_actual_f',
  'v2_spread_per_warmup_f',
  'v2_humidity_warmup_interaction',
  'v3_high_so_far_above_current_f',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_high_so_far_minus_lag_1d_f',
  'v3_high_so_far_minus_7d_actual_f',
  'v3_remaining_warmup_per_spread_f',
  'v3_humidity_remaining_warmup_interaction',
  'v4_forecast_precip_total_mean_mm',
  'v4_forecast_precip_total_max_mm',
  'v4_forecast_precip_total_spread_mm',
  'v4_forecast_precip_max_1h_mean_mm',
  'v4_forecast_precip_hours_mean',
  'v4_forecast_precip_intensity_mean',
  'v4_forecast_precip_intensity_max',
  'v4_any_forecast_precip',
  'v4_all_forecast_precip',
  'v4_observed_precip_any',
  'v4_observed_precip_recent_mm_est',
  'v4_forecast_total_minus_observed_recent_mm',
  'v4_forecast_observed_precip_match',
  'v4_forecast_wet_observed_dry',
  'v4_observed_wet_forecast_dry',
  'v4_precip_humidity

## Data Availability


In [5]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
18,KMIA,gfs,1998,2021-01-01,2026-06-21
19,KMIA,hrrr,1998,2021-01-01,2026-06-21
20,KMIA,nbm,1997,2021-01-01,2026-06-21


## Model Scores


In [6]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v11",
    target_mode="remaining_warmup",
    target_source="settlement_first",
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11_settlement",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v11_settlement/KMIA_optuna.sqlite3')

In [7]:
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-07-07 00:26:39,279] A new study created in RDB with name: KMIA_v11_remaining_warmup_base_xgboost_mae_f
[I 2026-07-07 00:26:44,500] Trial 0 finished with value: 1.112935154482952 and parameters: {'n_estimators': 1342, 'learning_rate': 0.19043899115982607, 'max_depth': 9, 'min_child_weight': 2.481040974867813, 'gamma': 2.340279606636548, 'subsample': 0.4513964382185317, 'colsample_bytree': 0.3877543479093296, 'reg_alpha': 2.478071022662141, 'reg_lambda': 0.6132587025321562}. Best is trial 0 with value: 1.112935154482952.
[I 2026-07-07 00:27:06,862] Trial 1 finished with value: 0.9775797256173215 and parameters: {'n_estimators': 2493, 'learning_rate': 0.001120367191095075, 'max_depth': 12, 'min_child_weight': 21.368329072358772, 'gamma': 3.185086660174142, 'subsample': 0.4681862286846154, 'colsample_bytree': 0.46921293140473197, 'reg_alpha': 4.476173538513514e-07, 'reg_lambda': 0.20253776634919213}. Best is trial 1 with value: 0.9775797256173215.
[I 2026-07-07 00:27:19,541] Trial 

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,731,0.932064,1.242867
1,validation_2024_2025,lightgbm,731,0.943641,1.252556
2,validation_2024_2025,catboost,731,0.932207,1.224902
3,validation_2024_2025,provider_mean,731,1.536686,1.918554
4,validation_2024_2025,provider_median,731,1.517929,1.908290
5,validation_2024_2025,nbm_raw,731,1.526456,1.932473
6,validation_2024_2025,hrrr_raw,731,1.795777,2.663658
7,validation_2024_2025,gfs_raw,731,2.368248,2.702361
8,test_2026,xgboost,172,0.938309,1.255135
9,test_2026,lightgbm,172,0.958528,1.280596


In [8]:
exported_weights = export_station_model_weights(
    project_root=PROJECT_ROOT,
    station_id=STATION_ID,
    artifact_dir=config.resolved_output_dir(),
    model_version=MODEL_VERSION,
    timing_mode=config.timing_mode,
    providers=tuple(config.providers),
    feature_version=config.effective_feature_version,
    optuna_metric=config.effective_optuna_metric,
    target_mode=config.effective_target_mode,
    target_source=config.effective_target_source,
    base_model_methods=tuple(config.effective_base_model_methods),
    stack_enabled=config.stack_enabled,
    source_pipeline="notebooks/experiments/station_stacking_v11_settlement",
)

exported_weights.bundle_path, exported_weights.manifest_path


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v11_settlement/model_weights/KMIA_station_high_regressor_v11_wunderground_settlement_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v11_settlement/model_weights/KMIA_station_high_regressor_v11_wunderground_settlement_stack.json'))

## V11 Feature Coverage


In [9]:
v11_feature_coverage = (
    result.features[V11_FEATURE_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

v11_feature_coverage


,feature,coverage_pct
0,v2_spread_per_warmup_f,100.000000
1,v2_morning_warmup_to_consensus_f,100.000000
2,v3_remaining_warmup_from_high_so_far_f,100.000000
3,v3_high_so_far_above_current_f,100.000000
4,v2_humidity_warmup_interaction,100.000000
5,v4_forecast_wet_observed_dry,100.000000
6,v4_forecast_observed_precip_match,100.000000
7,v4_all_forecast_precip,100.000000
8,v3_humidity_remaining_warmup_interaction,100.000000
9,v3_remaining_warmup_per_spread_f,100.000000


In [10]:
result.feature_columns.loc[result.feature_columns["feature"].isin(V11_FEATURE_COLUMNS)]


,feature,kind
20,observed_temp_change_last_1h_f,numeric
21,observed_temp_change_last_3h_f,numeric
22,observed_morning_warmup_rate_f_per_hour,numeric
23,observed_high_so_far_change_since_9am_f,numeric
221,v2_recent_heat_anomaly_f,numeric
222,v2_recent_heat_momentum_f,numeric
223,v2_morning_warmup_to_consensus_f,numeric
224,v2_consensus_minus_7d_actual_f,numeric
225,v2_spread_per_warmup_f,numeric
226,v2_humidity_warmup_interaction,numeric


## Dropped Feature Check


In [11]:
dropped_present = result.feature_columns.loc[
    result.feature_columns["feature"].isin(V11_DROPPED_FEATURE_COLUMNS)
]

dropped_present


,feature,kind


## Morning Trend Coverage


In [12]:
trend_coverage = (
    result.features[TREND_COLUMNS]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)

trend_coverage


,feature,coverage_pct
0,observed_temp_change_last_1h_f,99.94995
1,observed_temp_change_last_3h_f,99.94995
2,observed_morning_warmup_rate_f_per_hour,99.94995
3,observed_high_so_far_change_since_9am_f,99.94995


## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
0,oof_2026,catboost,172,137,79.651163
3,oof_2026,lightgbm,172,135,78.488372
7,oof_2026,ridge_stack,172,133,77.325581
8,oof_2026,xgboost,172,133,77.325581
2,oof_2026,hrrr_raw,172,114,66.279070
4,oof_2026,nbm_raw,172,109,63.372093
6,oof_2026,provider_median,172,105,61.046512
5,oof_2026,provider_mean,172,92,53.488372
1,oof_2026,gfs_raw,172,30,17.441860
9,validation_2024_2025,catboost,731,595,81.395349


## Version Comparison


In [14]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
    ("v6", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v6"),
    ("v7", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v7"),
    ("v8", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v8"),
    ("v9", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9"),
    ("v10", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v10"),
    ("v11", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v11"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,catboost,104,1.550583,2.063820,v5
1,test_2026,ridge_stack,104,1.552819,2.095250,v5
2,test_2026,catboost,104,1.563938,2.156057,v6
3,test_2026,lightgbm,107,1.580786,2.439781,v7
4,test_2026,ridge_stack,104,1.591358,2.159365,v6
...,...,...,...,...,...,...
72,validation_2024_2025,gfs_raw,664,3.540756,4.037614,v2
73,validation_2024_2025,gfs_raw,664,3.540756,4.037614,v3
74,validation_2024_2025,gfs_raw,562,3.592958,4.064970,v5
75,validation_2024_2025,gfs_raw,562,3.592958,4.064970,v6


## 2026 OOF Weather Brackets


In [ ]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,172,0.938309,1.255135,54.651163
1,lightgbm,172,0.958528,1.280596,54.069767
2,catboost,172,0.932000,1.247739,58.139535
3,ridge_stack,172,0.940079,1.246099,54.651163
4,provider_mean,172,1.563641,1.866075,30.813953
5,provider_median,172,1.390211,1.798649,40.116279
6,nbm_raw,172,1.399121,1.845821,36.046512
7,hrrr_raw,172,1.299044,1.736598,47.093023
8,gfs_raw,172,2.818558,3.109139,11.046512


: 